In [1]:
%pip install sqlalchemy "psycopg[binary]"

   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.7 MB 435.7 kB/s eta 0:00:09
   - -------------------------------------- 0.1/3.7 MB 1.1 MB/s eta 0:00:04
   --------- ------------------------------ 0.9/3.7 MB 5.7 MB/s eta 0:00:01
   ----------------------- ---------------- 2.2/3.7 MB 10.6 MB/s eta 0:00:01
   ---------------------------------------  3.6/3.7 MB 14.3 MB/s eta 0:00:01
   ---------------------------------------  3.7/3.7 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------  3.7/3.7 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------  3.7/3.7 MB 13.7 MB/s eta 0:00:01
   ---------------------------------------- 3.7/3.7 MB 8.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/215.5 kB ? eta -:--:--
   --------------------------------------- 215.5/215.5 kB 13.7 MB/s eta 0:00:00
Note: you may need t

In [2]:
# connecting python to postgreSQL

from sqlalchemy import create_engine, URL, text
from getpass import getpass

# Database connection details
db_user = "postgres"
db_password = getpass("Enter PostgreSQL password: ")
db_host = "localhost"
db_port = 5432
db_name = "customer_churn_db"

# Create connection URL
database_url = URL.create(
    drivername="postgresql+psycopg",
    username=db_user,
    password=db_password,
    host=db_host,
    port=db_port,
    database=db_name
)

# Create SQLAlchemy engine
engine = create_engine(database_url)


Enter PostgreSQL password:  ········


In [3]:
# testing the database connection

try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT current_database(), current_user, version();")
        )
        row = result.fetchone()

        print("Database connection successful!")
        print("Database:", row[0])
        print("User:", row[1])
        print("PostgreSQL:", row[2].split(",")[0])

except Exception as e:
    print("Connection failed:", e)


Database connection successful!
Database: customer_churn_db
User: postgres
PostgreSQL: PostgreSQL 18.6 on x86_64-windows


In [4]:
# verifying the connection from PostgreSQL

with engine.connect() as connection:
    result = connection.execute(
        text("SELECT datname FROM pg_database WHERE datname = 'customer_churn_db';")
    )
    print(result.fetchone())

('customer_churn_db',)


In [5]:
# loading the dataset into notebook

import pandas as pd

# Update this path to where your CSV is saved
csv_path = "data/Telco-Customer-Churn.csv"

df = pd.read_csv(csv_path, dtype={"TotalCharges": "string"})

print("Dataset shape:", df.shape)
print(df.head())


Dataset shape: (7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingM

In [6]:

# Convert TotalCharges to numeric, preserving missing values
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
)

print("TotalCharges dtype:", df["TotalCharges"].dtype)
print("Missing TotalCharges:", df["TotalCharges"].isna().sum())
print("Total records:", len(df))


TotalCharges dtype: Float64
Missing TotalCharges: 11
Total records: 7043


In [7]:
# load the dataframe into PostgreSQL

from sqlalchemy import types

df.to_sql(
    name="telco_customers",
    con=engine,
    schema="public",
    if_exists="fail",
    index=False,
    dtype={
        "customerID": types.String(20),
        "SeniorCitizen": types.Integer(),
        "tenure": types.Integer(),
        "MonthlyCharges": types.Numeric(10, 2),
        "TotalCharges": types.Numeric(12, 2)
    },
    chunksize=1000
)

print("Data loaded into PostgreSQL successfully!")


Data loaded into PostgreSQL successfully!


In [8]:
# verify the imported data using SQL

from sqlalchemy import text

query = """
SELECT
    COUNT(*) AS total_customers,
    COUNT(DISTINCT "customerID") AS unique_customers,
    COUNT(*) FILTER (
        WHERE "TotalCharges" IS NULL
    ) AS missing_total_charges
FROM public.telco_customers;
"""

with engine.connect() as connection:
    result = connection.execute(text(query))
    print(result.fetchone())


(7043, 7043, 11)
